In [8]:
# toy_ngafid.py
# Create a toy NGAFID-like dataset for quick MAE prototyping (no large downloads).

from __future__ import annotations
import math, random, os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value

# ----------------------------
# 1) Schema (edit as you like)
# ----------------------------
# Typical NGAFID-style columns
COLUMNS = [
    "timestamp",  # string timestamp or seconds since start
    "altmsl",     # altitude (ft)
    "ias",        # indicated airspeed (knots)
    "vspd",       # vertical speed (ft/min)
    "pitch",      # degrees
    "roll",       # degrees
    "hdg",        # heading (deg)
    # Add any extras you might see in shards:
    # "VAL", "VCDI",
]

# Hugging Face feature types (keep timestamp as string; rest float32)
HF_FEATURES = Features({
    "timestamp": Value("string"),
    **{c: Value("float32") for c in COLUMNS if c != "timestamp"},
})

# --------------------------------------
# 2) Random generators (realistic-ish)
# --------------------------------------
def sample_flight(n_rows: int, start_time_s: int = 0) -> pd.DataFrame:
    """
    Create a single synthetic flight time series with n_rows.
    Values are crude but bounded to plausible avionics ranges.
    """
    t = np.arange(start_time_s, start_time_s + n_rows, dtype=np.int64)

    # altitude: climb, cruise, descent pattern
    climb = np.clip(np.linspace(0, 12000, n_rows//3), 0, None)
    cruise = np.full(n_rows//3, 12000.0)
    descent = np.clip(np.linspace(12000, 1000, n_rows - 2*(n_rows//3)), 0, None)
    alt = np.concatenate([climb, cruise, descent])
    alt += np.random.normal(0, 80, size=n_rows)

    # airspeed: ramp up then stable with noise (knots)
    ias = np.clip(140 + 20*np.tanh(np.linspace(-2, 2, n_rows)) + np.random.normal(0, 4, n_rows), 60, 220)

    # vertical speed (ft/min): noisy, around 0 in cruise
    vspd = np.random.normal(0, 200, n_rows)
    vspd[: n_rows//3] += 800   # climb bias
    vspd[-n_rows//3 :] -= 800  # descent bias
    vspd = np.clip(vspd, -2000, 2000)

    # pitch/roll in degrees
    pitch = np.clip(np.random.normal(2, 3, n_rows), -10, 15)   # gentle
    roll  = np.clip(np.random.normal(0, 10, n_rows), -45, 45)  # bank
    # heading: slow random walk in [0, 360)
    hdg = np.cumsum(np.random.normal(0, 1, n_rows)) % 360

    df = pd.DataFrame({
        "timestamp": pd.to_datetime(t, unit="s").astype(str),
        "altmsl": alt.astype("float32"),
        "ias": ias.astype("float32"),
        "vspd": vspd.astype("float32"),
        "pitch": pitch.astype("float32"),
        "roll": roll.astype("float32"),
        "hdg": hdg.astype("float32"),
    })

    return df

# --------------------------------------------------------
# 3) Build a toy dataset (in-memory and/or write shards)
# --------------------------------------------------------
def make_toy_ngafid_dataset(
    num_flights: int = 20,
    rows_per_flight: Tuple[int, int] = (2000, 6000),  # flights vary in length
    write_shards_like_hf: bool = False,
    out_dir: str | Path = "toy_ngafid",
) -> DatasetDict:
    """
    Create a toy dataset with multiple flights. Returns a DatasetDict with a 'train' split.
    If write_shards_like_hf=True, also writes CSV shards to toy_ngafid/flights/*.csv to mirror HF layout.
    """
    rng = np.random.default_rng(0)

    all_frames: List[pd.DataFrame] = []
    out_dir = Path(out_dir)
    flights_dir = out_dir / "flights"
    if write_shards_like_hf:
        flights_dir.mkdir(parents=True, exist_ok=True)

    for i in range(num_flights):
        n_rows = int(rng.integers(rows_per_flight[0], rows_per_flight[1]))
        df = sample_flight(n_rows, start_time_s=i * 10_000)
        # Optional light missingness (simulate sensor dropouts)
        for col in ["altmsl", "ias", "vspd", "pitch", "roll", "hdg"]:
            mask = rng.random(n_rows) < 0.005  # 0.5% NaNs
            df.loc[mask, col] = np.nan

        if write_shards_like_hf:
            # write a shard named like the real dataset
            shard_path = flights_dir / f"flight_{i:05d}.csv"
            df.to_csv(shard_path, index=False)

        all_frames.append(df)

    big_df = pd.concat(all_frames, ignore_index=True)

    # Ensure only the declared columns exist and types match
    big_df = big_df[[c for c in COLUMNS]]  # drop any extras accidentally added

    ds = Dataset.from_pandas(big_df, features=HF_FEATURES, preserve_index=False)
    return DatasetDict({"train": ds})

# -----------------------
# 4) Quick usage example
# -----------------------
if __name__ == "__main__":
    toy = make_toy_ngafid_dataset(
        num_flights=10,
        rows_per_flight=(1500, 3000),
        write_shards_like_hf=True,   # set False if you only want in-memory HF dataset
        out_dir="toy_ngafid",
    )

    print(toy)               # DatasetDict with 'train'
    print(toy.shape)
    print(toy["train"])      # size (rows) and columns
    print(toy["train"].shape)      # size (rows) and columns
    print(toy["train"][0])   # one row sample
    # You can now plug toy["train"] into your MAE dataloader.


DatasetDict({
    train: Dataset({
        features: ['timestamp', 'altmsl', 'ias', 'vspd', 'pitch', 'roll', 'hdg'],
        num_rows: 23880
    })
})
{'train': (23880, 7)}
Dataset({
    features: ['timestamp', 'altmsl', 'ias', 'vspd', 'pitch', 'roll', 'hdg'],
    num_rows: 23880
})
(23880, 7)
{'timestamp': '1970-01-01 00:00:00', 'altmsl': 109.2603988647461, 'ias': 132.31459045410156, 'vspd': 890.3931884765625, 'pitch': 2.7590396404266357, 'roll': -12.099011421203613, 'hdg': 358.62335205078125}


In [12]:
from datasets import load_dataset
from itertools import islice

ds = load_dataset(
    "csv",
    data_files="hf://datasets/CDuong04/NGAFID-LOCI-GATS-Data/preprocessed_data/train/*.csv",
    streaming=True,
)["train"]

all_cols = set()
N = 2000  # scan first N rows (increase if needed)
for row in islice(ds, N):
    all_cols.update(row.keys())

print("Discovered columns:", len(all_cols))
print(sorted(all_cols))


Discovered columns: 44
['altagl', 'altb', 'altgps', 'altmsl', 'altmsllagdiff', 'amp1', 'aoasimple', 'baroa', 'crs', 'densityratio', 'e1egt1', 'e1egt2', 'e1egt3', 'e1egt4', 'e1egtdivergence', 'e1fflow', 'e1oilp', 'e1oilt', 'e1rpm', 'fqtyl', 'fqtyr', 'gndspd', 'hdg', 'hplfd', 'hplwas', 'ias', 'latac', 'magvar', 'normac', 'oat', 'pitch', 'roll', 'stallindex', 'tas', 'totalfuel', 'trk', 'trueairspeed(ft/min)', 'volt1', 'vplwas', 'vspd', 'vspdcalculated', 'vspdg', 'wnddr', 'wndspd']


In [13]:
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value

# config
N_FLIGHTS = 50
T = 9999      # timesteps per flight
F = 44        # features per timestep

# give the features stable names; replace with real names later
F_NAMES = [f"feat_{i:02d}" for i in range(F)]
FEATURES = Features({"timestamp": Value("int64"), **{f: Value("float32") for f in F_NAMES}})

# make one big flat table (flight_id, step, features...) to mimic CSV structure
rows = []
rng = np.random.default_rng(0)
for flight in range(N_FLIGHTS):
    # simple random walk-ish signals to look time-seriesy
    base = rng.normal(0, 1, (T, F)).cumsum(axis=0) / 50.0
    # timestamp = just an integer step or seconds since start
    ts = np.arange(T, dtype=np.int64)
    df = pd.DataFrame({"timestamp": ts})
    for j, name in enumerate(F_NAMES):
        df[name] = base[:, j].astype("float32")
    df["flight_id"] = flight  # optional if you need grouping
    rows.append(df)

big = pd.concat(rows, ignore_index=True)

# drop flight_id from features if you don’t want it in the model input
big = big[["timestamp"] + F_NAMES]

toy = Dataset.from_pandas(big, features=FEATURES, preserve_index=False)
toy = DatasetDict({"train": toy})

print(toy)          # shows total rows and columns (timestamp + 44)
print(toy["train"]) # schema
print(toy["train"][0])


DatasetDict({
    train: Dataset({
        features: ['timestamp', 'feat_00', 'feat_01', 'feat_02', 'feat_03', 'feat_04', 'feat_05', 'feat_06', 'feat_07', 'feat_08', 'feat_09', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18', 'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24', 'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30', 'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36', 'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42', 'feat_43'],
        num_rows: 499950
    })
})
Dataset({
    features: ['timestamp', 'feat_00', 'feat_01', 'feat_02', 'feat_03', 'feat_04', 'feat_05', 'feat_06', 'feat_07', 'feat_08', 'feat_09', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18', 'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24', 'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30', 'feat_31', 'feat_32', 'feat_33', '